In [33]:
import pandas as pd
from pathlib import Path

In [32]:


# Read just the one file we need
# nrows is gone — we want all rows this time
# low_memory=False fixes the mixed type warning
csv_path = r"J:\- Macros\AI-LaneDetector\lane_analysis\coordinates_report_with_lane_combined_with_ignore_full1.csv"

df = pd.read_csv(csv_path, low_memory=False)
# reads the whole CSV into a dataframe
# Java equivalent: reading entire file into a List<Row>

print(f"Total rows: {len(df):,}")

print(f"\nLane distribution:")
print(df["Lane"].value_counts().to_string())

print(f"\nIgnore distribution:")
print(df["Ignore"].value_counts().to_string())
# now we can see what the Ignore column contains

print(f"\nSample rows:")
pd.set_option('display.max_columns', None)
# show ALL columns, pandas hides some by default when there are many
print(df.head(3).to_string())

KeyboardInterrupt: 

In [21]:
SKIP_FILES = [
    "per_project_stats.csv",
    "per_session_stats.csv",
]

for csv_file in Path(csv_folder).glob("*.csv"):

    if csv_file.name in SKIP_FILES:
        continue

    # nrows=5 = only read first 5 rows — just to check column names
    # Java equivalent: like reading only the header line of a CSV
    df_peek = pd.read_csv(csv_file, nrows=5, low_memory=False)

    print(f"{csv_file.name}:")
    print(f"  Columns: {df_peek.columns.tolist()}")
    # just show the column names, nothing else

coordinates_report_combined_with_ignore.csv:
  Columns: ['Filename', 'Latitude', 'Longitude', 'TestDateUTC', 'Bearing', 'Lane', 'Filepath', '_source_folder', '_session', '_source_file', 'Longitudinal Crack', 'Transverse Crack', 'Alligator Crack', 'Potholes', 'DistressDetected', 'MaxDistressConfidence', 'DetectionSource', 'Ignore', '_matched_session']
coordinates_report_with_lane_combined.csv:
  Columns: ['SourceFolder', 'Filename', 'Latitude', 'Longitude', 'TestDateUTC', 'Bearing', 'Filepath', 'Timestamp', 'Plate', 'Lane']
coordinates_report_with_lane_combined_with_ignore.csv:
  Columns: ['SourceFolder', 'Filename', 'Latitude', 'Longitude', 'TestDateUTC', 'Bearing', 'Filepath', 'Timestamp', 'Plate', 'Lane', 'Ignore']
coordinates_report_with_lane_combined_with_ignore_full.csv:
  Columns: ['Filename', 'Latitude', 'Longitude', 'TestDateUTC', 'Bearing', 'Lane', 'Filepath', '_source_folder', '_session', '_source_file', 'Ignore', '_matched_session']
coordinates_report_with_lane_combined_with

In [16]:
for csv_file in Path(csv_folder).glob("*.csv"):
    if csv_file.name in SKIP_FIILES:
        print(f"SKIPPING {csv_file.name}")
        continue
        df = pd.read_csv(csv_file, low_memory = False)
    if "filepath" not in df.columns or "Lane" not in df.columns:
        print(f"Skipping {csv_file.name} - missing FilePath or lane column")
        continue
    df["source_csv"] = csv_file.name
    all_dfs.append(df)
    print(f"{csv_file.name}: {len(df):, } rows")


Skipping coordinates_report_combined_with_ignore.csv - missing FilePath or lane column
Skipping coordinates_report_with_lane_combined.csv - missing FilePath or lane column
Skipping coordinates_report_with_lane_combined_with_ignore.csv - missing FilePath or lane column
Skipping coordinates_report_with_lane_combined_with_ignore_full.csv - missing FilePath or lane column
Skipping coordinates_report_with_lane_combined_with_ignore_full1.csv - missing FilePath or lane column
Skipping merged_laneFixes.csv - missing FilePath or lane column
Skipping per_project_stats.csv - missing FilePath or lane column
Skipping per_session_stats.csv - missing FilePath or lane column
Skipping SK1.csv - missing FilePath or lane column


In [23]:
# Do ignored frames also have a Lane value?
# Or is Lane empty when Ignore=1?
print(df[df["Ignore"] == 1.0][["Lane", "Ignore"]].value_counts().to_string())
# filters rows where Ignore equals 1.0
# then counts combinations of Lane and Ignore values

Lane  Ignore
1     1.0       66927
SK1   1.0        5681
SK2   1.0         420
2     1.0         213
SK3   1.0         129


In [25]:
print(df[df["Lane"] == "BK4"]["Filepath"].head(3).to_string())
# shows filepaths of BK4 rows so boss can look at the actual images

463820    J:\Testing\250362.05 AT_Data_Collection_25_LMD...
463821    J:\Testing\250362.05 AT_Data_Collection_25_LMD...
463822    J:\Testing\250362.05 AT_Data_Collection_25_LMD...


In [26]:
# Ignore = 1.0 rows → excluded from training entirely
# Ignore = NaN  rows → use Lane value for training

# So we DON'T need "Ignore" as a class
# We just filter those rows out before training

df_clean = df[df["Ignore"] != 1.0].copy()
# != means "not equal to"
# Java equivalent: df.stream().filter(r -> !r.getIgnore().equals(1.0))

print(f"Total rows:           {len(df):,}")
print(f"Ignore rows removed:  {df['Ignore'].eq(1.0).sum():,}")
print(f"Remaining for training: {len(df_clean):,}")

print(f"\nLane distribution after removing ignored frames:")
print(df_clean["Lane"].value_counts().to_string())

Total rows:           2,840,370
Ignore rows removed:  73,370
Remaining for training: 2,767,000

Lane distribution after removing ignored frames:
Lane
1      2651847
2        94966
3        11406
SK1       6047
4          478
SK2        149
BK4         88
SK3         19


In [34]:
# ---- Extract segment from Filepath ----
df_clean["SourceFolder"] = df_clean["Filepath"].str.extract(r"Testing\\([^\\]+)\\")
# str.extract() pulls text matching a regex pattern from a string column
# r"Testing\\([^\\]+)\\" captures everything between "Testing\" and the next "\"
# which is the project/segment folder name
# Java equivalent: Pattern.compile("Testing\\\\([^\\\\]+)\\\\")
#                         .matcher(filepath).group(1)

print("Segments found:")
print(df_clean["SourceFolder"].value_counts().to_string())

print(f"\nNull segments: {df_clean["SourceFolder"].isna().sum():,}")
# checks if any filepaths didn't match the pattern

Segments found:
SourceFolder
250816 SouthlandDC_NetworkLMD_25             1304825
250362.01 AT_Data_Collection_25_LMD           347290
250846 DunedinCC LMD Network26                332279
250819 SelwynDC_NetworkLMD_25_Demo            190566
250724 WaitakiDC_Network_LMD25                178309
250362.05 AT_Data_Collection_25_LMD_North     162270
250728 InvercargillCC_NetworkLMD 25           108943
250357 UHCC_25_LMD                             87014
250821 KaikouraDC_Network25_LMD Demo           53248

Null segments: 0


In [27]:
# Classes we are confident about
EXCLUDE_LANES = ["BK4", "SK3", "SK2"]
# list of lane codes to remove
# revisit SK2 after talking to boss

LANE_CLASSES = {
    "1":   0,
    "2":   1,
    "3":   2,
    "SK1": 3,
    "4":   4,
}
# dictionary mapping lane code string → integer index
# the model outputs numbers, not strings
# so "1" → 0, "2" → 1 etc.
# Java equivalent: Map<String, Integer> laneClasses = new HashMap<>();

IDX_TO_LANE = {v: k for k, v in LANE_CLASSES.items()}
# reverses the dictionary — maps integer back to string
# so 0 → "1", 1 → "2" etc.
# used when converting model output back to lane code
# {v: k for k, v in dict.items()} = dictionary comprehension
# Java equivalent: a reverse loop building a new HashMap

# Apply to dataframe
df_clean = df_clean[~df_clean["Lane"].isin(EXCLUDE_LANES)].copy()
# ~ = NOT operator (inverts True/False)
# .isin() = checks if value is in the list
# together: keep rows where Lane is NOT in the exclude list
# Java equivalent:
# rows.stream()
#   .filter(r -> !EXCLUDE_LANES.contains(r.getLane()))
#   .collect(Collectors.toList())

df_clean = df_clean[df_clean["Lane"].isin(LANE_CLASSES)].copy()
# extra safety — only keep rows with known lane codes

print(f"Final clean rows: {len(df_clean):,}")
print(f"\nFinal lane distribution:")
for lane, idx in LANE_CLASSES.items():
    count = (df_clean["Lane"] == lane).sum()
    pct   = count / len(df_clean) * 100
    print(f"  {lane:>4} (class {idx}): {count:>10,} rows  ({pct:.2f}%)")

Final clean rows: 2,764,744

Final lane distribution:
     1 (class 0):  2,651,847 rows  (95.92%)
     2 (class 1):     94,966 rows  (3.43%)
     3 (class 2):     11,406 rows  (0.41%)
   SK1 (class 3):      6,047 rows  (0.22%)
     4 (class 4):        478 rows  (0.02%)


In [1]:
from sklearn.model_selection import train_test_split
# sklearn = scikit-learn, Python's machine learning utility library
# train_test_split = splits data into two parts randomly
# Java equivalent: no direct equivalent, would need manual implementation

# ---- Strategy ----
# Common classes (Lane 1, 2, 3, SK1) → split by segment as before
# Rare classes (Lane 4) → stratified split to guarantee representation

# Separate rare and common classes
RARE_CLASSES   = ["4"]
# Lane 4 only has 478 rows — too few to rely on segment split

COMMON_CLASSES = ["1", "2", "3", "SK1"]

df_common = df_clean[df_clean["Lane"].isin(COMMON_CLASSES)].copy()
# keep only rows where Lane is in COMMON_CLASSES
# Java equivalent: rows.stream()
#                      .filter(r -> COMMON_CLASSES.contains(r.getLane()))

df_rare   = df_clean[df_clean["Lane"].isin(RARE_CLASSES)].copy()
# keep only rare class rows

print(f"Common class rows: {len(df_common):,}")
print(f"Rare class rows:   {len(df_rare):,}")

# ---- Split common classes by segment (same as before) ----
train_common = df_common[
    ~df_common["SourceFolder"].isin(TEST_SEGMENTS + VAL_SEGMENTS)
].copy()

val_common = df_common[
    df_common["SourceFolder"].isin(VAL_SEGMENTS)
].copy()

test_common = df_common[
    df_common["SourceFolder"].isin(TEST_SEGMENTS)
].copy()

# ---- Split rare classes by stratified random split ----
# stratify=df_rare["Lane"] = ensures each split has proportional
# representation of each rare class
# e.g. if Lane 4 has 478 rows:
# train gets ~80% = 382 rows
# val   gets ~10% =  48 rows
# test  gets ~10% =  48 rows

train_rare_val, test_rare = train_test_split(
    df_rare,
    test_size=0.1,          # 10% goes to test
    random_state=42,        # fixed seed = reproducible split every time
                            # Java equivalent: new Random(42)
    stratify=df_rare["Lane"]
)

train_rare, val_rare = train_test_split(
    train_rare_val,
    test_size=0.1,          # 10% of remaining goes to val
    random_state=42,
    stratify=train_rare_val["Lane"]
)

# ---- Combine common + rare for each split ----
train_df = pd.concat([train_common, train_rare], ignore_index=True)
val_df   = pd.concat([val_common,   val_rare],   ignore_index=True)
test_df  = pd.concat([test_common,  test_rare],  ignore_index=True)
# pd.concat() stacks dataframes vertically
# ignore_index=True resets row numbers from 0
# Java equivalent: list1.addAll(list2)

# ---- Shuffle each split ----
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = val_df.sample(frac=1,   random_state=42).reset_index(drop=True)
test_df  = test_df.sample(frac=1,  random_state=42).reset_index(drop=True)
# .sample(frac=1) = shuffle all rows randomly
# frac=1 means "sample 100% of rows" = shuffle in place
# without shuffle, all Lane 4 rows would be bunched at the end
# Java equivalent: Collections.shuffle(list, new Random(42))
# reset_index(drop=True) = reset row numbers after shuffle
# drop=True = don't keep old row numbers as a column

# ---- Print final summary ----
print(f"\nFinal splits:")
print(f"Train: {len(train_df):,} rows")
print(f"Val:   {len(val_df):,} rows")
print(f"Test:  {len(test_df):,} rows")

for split_name, split_df in [("Train", train_df),
                               ("Val",   val_df),
                               ("Test",  test_df)]:
    # iterating over a list of tuples
    # each iteration unpacks into (split_name, split_df)
    # Java equivalent: for (Map.Entry<String, DataFrame> entry : splits.entrySet())
    print(f"\n{split_name} lane distribution:")
    for lane, idx in LANE_CLASSES.items():
        count = (split_df["Lane"] == lane).sum()
        pct   = count / len(split_df) * 100
        print(f"  {lane:>4}: {count:>10,} ({pct:.2f}%)")

# ---- Save ----
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv",     index=False)
test_df.to_csv("test.csv",   index=False)
print("\nSaved: train.csv, val.csv, test.csv")

NameError: name 'df_clean' is not defined

In [1]:
import pandas as pd

print("Loading new CSV...")
df = pd.read_csv(
    r"J:\- Macros\AI-LaneDetector\lane_analysis\coordinates_report_merged_fix_folder1.csv",
    low_memory=False
)
print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLane distribution:")
print(df["Lane"].value_counts().to_string())

Loading new CSV...
Total rows: 12,474,809
Columns: ['Filename', 'Latitude', 'Longitude', 'TestDateUTC', 'Bearing', 'Filepath', 'Lane', 'Ignore']

Lane distribution:
Lane
1      11140698
2       1059049
3        155011
SK1       59193
-1        22030
4         14877
TK1       10952
TM1        6828
SK2        2824
TK2        1029
TM2         589
BK4         542
5           397
SK3         288
0           279
TK3         114
SM1          51
TM3          45
B3           13


In [2]:
import pandas as pd

print("Loading new CSV...")
df = pd.read_csv(
    r"J:\- Macros\AI-LaneDetector\lane_analysis\coordinates_report_merged_fix_folder1.csv",
    low_memory=False
)
print(f"Total rows: {len(df):,}")

# keep only Lane 1, 2, 3, SK1
valid_lanes = ["1", "2", "3", "SK1"]
df = df[df["Lane"].astype(str).isin(valid_lanes)].copy()
df["Lane"] = df["Lane"].astype(str)
print(f"After filtering: {len(df):,}")

# 10% sample of each class - balanced
lane1 = df[df["Lane"] == "1"].sample(317714, random_state=42)
lane2 = df[df["Lane"] == "2"].sample(105904, random_state=42)
lane3 = df[df["Lane"] == "3"].sample(15501,  random_state=42)
sk1   = df[df["Lane"] == "SK1"].sample(5919,  random_state=42)

balanced = pd.concat([lane1, lane2, lane3, sk1]).sample(frac=1, random_state=42)
balanced = balanced.reset_index(drop=True)

print(f"\nBalanced dataset:")
print(f"Total: {len(balanced):,}")
print(balanced["Lane"].value_counts().to_string())

# split into 10 chunks
chunk_size = len(balanced) // 10
print(f"\nSplitting into 10 chunks of ~{chunk_size:,} rows each...")

from pathlib import Path
output_dir = Path("Data/chunks_v2")
output_dir.mkdir(exist_ok=True)

for i in range(10):
    start = i * chunk_size
    end   = start + chunk_size if i < 9 else len(balanced)
    chunk = balanced.iloc[start:end].copy()

    output_path = output_dir / f"train_chunk_v2_{i+1:02d}.csv"
    chunk.to_csv(output_path, index=False)
    print(f"Chunk {i+1:02d}: {len(chunk):,} rows → {output_path}")
    print(f"  {chunk['Lane'].value_counts().to_string()}")

print("\nDone! All chunks saved to Data/chunks_v2/")

Loading new CSV...
Total rows: 12,474,809
After filtering: 12,413,951

Balanced dataset:
Total: 445,038
Lane
1      317714
2      105904
3       15501
SK1      5919

Splitting into 10 chunks of ~44,503 rows each...
Chunk 01: 44,503 rows → Data\chunks_v2\train_chunk_v2_01.csv
  Lane
1      31688
2      10675
3       1550
SK1      590
Chunk 02: 44,503 rows → Data\chunks_v2\train_chunk_v2_02.csv
  Lane
1      31719
2      10597
3       1580
SK1      607
Chunk 03: 44,503 rows → Data\chunks_v2\train_chunk_v2_03.csv
  Lane
1      31683
2      10738
3       1529
SK1      553
Chunk 04: 44,503 rows → Data\chunks_v2\train_chunk_v2_04.csv
  Lane
1      31713
2      10634
3       1550
SK1      606
Chunk 05: 44,503 rows → Data\chunks_v2\train_chunk_v2_05.csv
  Lane
1      31969
2      10394
3       1566
SK1      574
Chunk 06: 44,503 rows → Data\chunks_v2\train_chunk_v2_06.csv
  Lane
1      31809
2      10532
3       1560
SK1      602
Chunk 07: 44,503 rows → Data\chunks_v2\train_chunk_v2_07.csv
  La

In [3]:
import pandas as pd
df = pd.read_csv("Data/chunks_v2/train_chunk_v2_02.csv", nrows=3)
for fp in df["Filepath"]:
    print(fp[:80])

J:\Testing\260078 GeoSolve_LMDDataCollection\Photos\QFM954-2026-06-30-NZST\QFM95
J:\Testing\260385 ChristchurchCC_LMD26\Photos\QFM954-2026-05-29-NZST\QFM954-2026
J:\Testing\250577 B_HASTINGS_LMD_25\Photos\QFM954-2025-09-02-NZST\QFM954-2025-09


In [ ]:
"""C:\Users\gd\New folder\project\geosolve_lanes\geosolve_lanes\.venv\Scripts\python.exe" "C:\Users\gd\New folder\project\geosolve_lanes\geosolve_lanes\evaluate.py"
using device: cpu
using phase 2 model
f
{'='*50}
 evaluation- test set
==================================================
 loading model from : checkpoints\best_phase2.pth
[Model] Backbone: efficientnet_b0
[Model] backbone output features: 1280
[Model] GPS processor: 5--> 64 features
[Model] Classifier: 1344 -> 4 classes
[Model] Output classes: 4
[Model] Loaded trained model from checkpoints\best_phase2.pth

Loading test data..
[Dataset] Loading Data\test.csv...
[Dataset] 87,014 valid images loaded (dropped ignore)
[Dataset] Lane distibution:
    1:     85,057 (97.8)
    2:      1,848 (2.1)
  SK1:         97 (0.1)
    3:         12 (0.0)
[DataLoader] Test: 87,014 images, 1,360 batches
Running evaluation..
[Dataset] WARNING : could non load J:/Testing/250357 UHCC_25_LMD/Photos/QFM954-2025-07-04-NZST/QFM954-2025-07-04-02-34-1/250357-2025-07-04-02-23-29-887-4113.49779S-17451.86791E-107.3-1-0-QFM954---F-.jpg
[ WARN:0@50.389] global loadsave.cpp:278 cv::findDecoder imread_('J:/Testing/250357 UHCC_25_LMD/Photos/QFM954-2025-07-04-NZST/QFM954-2025-07-04-02-34-1/250357-2025-07-04-02-23-29-887-4113.49779S-17451.86791E-107.3-1-0-QFM954---F-.jpg'): can't open/read file: check file path/integrity"""
